<a href="https://colab.research.google.com/github/jp-mmoura/PPA1-IA-02_03_26/blob/main/IA_Aula2_Case_Regras_9P.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inteligência Artificial (9º período) — Aula 2
## Notebook: Modelagem de Problemas + Dados + Sistema de Decisão Baseado em Regras (Python)

**Objetivo:** consolidar a ideia de que IA começa pela formulação do problema e pela qualidade dos dados — e implementar um *sistema de decisão baseado em regras*.

> **Aviso didático:** este notebook foca em IA simbólica (regras) como ponte para ML nas próximas aulas.


## 1) Warm-up: o que é um “problema de IA”?
Um problema (bem definido) pode ser descrito por:
- **Estado** (como o mundo está)
- **Ações** (o que podemos fazer)
- **Objetivo** (o que queremos alcançar)
- **Critério de desempenho/custo**

Nesta aula, vamos modelar um problema de **decisão** (ação recomendada) com base em variáveis observáveis (dados).


## 2) Case técnico: decisão de reposição (simplificado)
Vamos simular um cenário de decisão:
- **Entradas (dados):** estoque, ROP (ponto de ressuprimento), demanda (média), variabilidade (desvio padrão), lead time, criticidade, etc.
- **Saída (ação):** `COMPRAR`, `AGUARDAR`, `URGENTE` (ou semelhante)

**Importante:** isso *não* é otimização de estoque completa. É um exemplo didático para:
- exercitar **modelagem**
- exercitar **regras** e **priorização**
- discutir **falhas** por dados ruins


In [1]:
from dataclasses import dataclass
from typing import Callable, Dict, List, Tuple
import math
import random
import pandas as pd


## 3) Gerando um dataset didático
Vamos criar uma base sintética com algumas variáveis típicas.
- Você pode alterar parâmetros e ver como isso muda decisões.


In [2]:
def make_dataset(n=60, seed=42):
    random.seed(seed)
    rows = []
    for i in range(n):
        sku = f"SKU-{i:03d}"
        rop = random.randint(20, 120)
        estoque = max(0, int(random.gauss(mu=rop, sigma=rop*0.35)))
        demanda_media = max(1, int(random.gauss(mu=rop/2, sigma=rop*0.15)))
        demanda_std = max(0, int(random.gauss(mu=demanda_media*0.25, sigma=max(1, demanda_media*0.10))))
        lead_time = random.choice([3, 5, 7, 10, 14, 21])
        criticidade = random.choice(["A", "B", "C"])  # A = mais crítico
        margem = round(random.uniform(0.05, 0.35), 2)
        rows.append({
            "sku": sku,
            "estoque": estoque,
            "rop": rop,
            "demanda_media": demanda_media,
            "demanda_std": demanda_std,
            "lead_time": lead_time,
            "criticidade": criticidade,
            "margem": margem,
        })
    return pd.DataFrame(rows)

df = make_dataset()
df.head(10)


,sku,estoque,rop,demanda_media,demanda_std,lead_time,criticidade,margem
0,SKU-000,145,101,66,16,3,C,0.27
1,SKU-001,106,89,56,17,3,A,0.12
2,SKU-002,53,84,32,6,10,A,0.18
3,SKU-003,25,55,27,6,5,C,0.18
4,SKU-004,82,55,44,9,3,B,0.30
5,SKU-005,109,97,48,13,10,C,0.09
6,SKU-006,85,68,38,4,7,C,0.11
7,SKU-007,13,28,16,4,7,A,0.31
8,SKU-008,23,32,19,5,7,A,0.25
9,SKU-009,89,109,76,14,14,C,0.10


## 4) Regras: como representar conhecimento
Uma regra pode ser vista como:
- **Condição** → **Ação**

Também podemos atribuir **prioridade** (ordem) e um texto de **justificativa**.


In [3]:
@dataclass
class Rule:
    name: str
    priority: int
    condition: Callable[[Dict], bool]
    action: str
    rationale: str

def apply_rules(row: Dict, rules: List[Rule]) -> Tuple[str, str, str]:
    """Aplica regras por ordem de prioridade (menor número = maior prioridade).
    Retorna (acao, regra_disparada, justificativa).
    """
    for r in sorted(rules, key=lambda x: x.priority):
        if r.condition(row):
            return r.action, r.name, r.rationale
    return "AGUARDAR", "DEFAULT", "Nenhuma regra disparou; manter monitoramento."


## 5) Definindo um conjunto de regras (didático)
Regras simples para ilustrar:
- `URGENTE` se estoque muito abaixo do ROP e criticidade alta
- `COMPRAR` se abaixo do ROP e variabilidade/lead time indicam risco
- caso contrário, `AGUARDAR`

Pergunta-chave: **onde essas regras falham?**


In [8]:
rules = [
    Rule(
        name="R0_INVALIDO",
        priority=0,
        condition=lambda r: (r["rop"] == 0),
        action="REVISAR_DADOS",
        rationale="ROP igual a zero indica dado inválido ou faltante. Necessário revisar."
    ),
    Rule(
        name="R1_URGENTE_CRITICO",
        priority=1,
        condition=lambda r: (r["criticidade"] == "A") and (r["estoque"] < 0.6 * r["rop"]),
        action="URGENTE",
        rationale="Item crítico (A) com estoque muito abaixo do ROP. Risco alto de ruptura."
    ),
    Rule(
        name="R2_COMPRAR_ABAIXO_ROP",
        priority=2,
        condition=lambda r: (r["estoque"] < r["rop"]) and (r["demanda_std"] > 0.2 * r["demanda_media"]),
        action="COMPRAR",
        rationale="Estoque abaixo do ROP e variabilidade elevada. Recomenda compra preventiva."
    ),
    Rule(
        name="R3_COMPRAR_LEADTIME_ALTO",
        priority=3,
        condition=lambda r: (r["estoque"] < r["rop"]) and (r["lead_time"] >= 14),
        action="COMPRAR",
        rationale="Estoque abaixo do ROP e lead time alto. Melhor comprar cedo para evitar ruptura."
    ),
    Rule(
        name="R4_COMPRAR_MARGEM_ALTA",
        priority=4,
        condition=lambda r: (r["margem"] >= 0.25) and (r["estoque"] < r["rop"]),
        action="COMPRAR",
        rationale="Margem de lucro alta e estoque abaixo do ROP. Comprar para garantir disponibilidade."
    ),
]

def decide(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    actions, fired, why = [], [], []
    for _, row in out.iterrows():
        a, f, w = apply_rules(row.to_dict(), rules)
        actions.append(a)
        fired.append(f)
        why.append(w)
    out["acao"] = actions
    out["regra"] = fired
    out["justificativa"] = why
    return out

df_dec = decide(df)
df_dec.head(12)

,sku,estoque,rop,demanda_media,demanda_std,lead_time,criticidade,margem,acao,regra,justificativa
0,SKU-000,145,101,66,16,3,C,0.27,AGUARDAR,DEFAULT,Nenhuma regra disparou; manter monitoramento.
1,SKU-001,106,89,56,17,3,A,0.12,AGUARDAR,DEFAULT,Nenhuma regra disparou; manter monitoramento.
2,SKU-002,53,84,32,6,10,A,0.18,AGUARDAR,DEFAULT,Nenhuma regra disparou; manter monitoramento.
3,SKU-003,25,55,27,6,5,C,0.18,COMPRAR,R2_COMPRAR_ABAIXO_ROP,Estoque abaixo do ROP e variabilidade elevada....
4,SKU-004,82,55,44,9,3,B,0.30,AGUARDAR,DEFAULT,Nenhuma regra disparou; manter monitoramento.
5,SKU-005,109,97,48,13,10,C,0.09,AGUARDAR,DEFAULT,Nenhuma regra disparou; manter monitoramento.
6,SKU-006,85,68,38,4,7,C,0.11,AGUARDAR,DEFAULT,Nenhuma regra disparou; manter monitoramento.
7,SKU-007,13,28,16,4,7,A,0.31,URGENTE,R1_URGENTE_CRITICO,Item crítico (A) com estoque muito abaixo do R...
8,SKU-008,23,32,19,5,7,A,0.25,COMPRAR,R2_COMPRAR_ABAIXO_ROP,Estoque abaixo do ROP e variabilidade elevada....
9,SKU-009,89,109,76,14,14,C,0.10,COMPRAR,R3_COMPRAR_LEADTIME_ALTO,Estoque abaixo do ROP e lead time alto. Melhor...


## 6) Análise rápida: distribuição das decisões


In [10]:
df_dec["acao"].value_counts()

,count
acao,
AGUARDAR,43
COMPRAR,11
URGENTE,6


## 7) Dados ruins: saneamento mínimo e impacto
Vamos inserir alguns erros típicos e aplicar um saneamento mínimo para evitar quebra.


In [12]:
df_bad = df.copy()
df_bad.loc[0, "rop"] = None
df_bad.loc[1, "demanda_media"] = 0
df_bad.loc[2, "lead_time"] = 999

def safe_row(row: Dict) -> Dict:
    r = dict(row)
    # defaults conservadores
    # Use Series to allow fillna, then convert back to scalar for the dict
    r["rop"] = pd.Series([r.get("rop")]).fillna(0).iloc[0]
    r["demanda_media"] = max(1, int(r.get("demanda_media") or 1))
    r["lead_time"] = min(int(r.get("lead_time") or 0), 60)  # cap didático
    return r

def decide_safe(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    actions, fired, why = [], [], []
    for _, row in out.iterrows():
        srow = safe_row(row.to_dict())
        a, f, w = apply_rules(srow, rules)
        actions.append(a)
        fired.append(f)
        why.append(w + " (após saneamento mínimo)")
    out["acao"] = actions
    out["regra"] = fired
    out["justificativa"] = why
    return out

df_bad_dec = decide_safe(df_bad)
df_bad_dec.head(6)

,sku,estoque,rop,demanda_media,demanda_std,lead_time,criticidade,margem,acao,regra,justificativa
0,SKU-000,145,NaN,66,16,3,C,0.27,REVISAR_DADOS,R0_INVALIDO,ROP igual a zero indica dado inválido ou falta...
1,SKU-001,106,89.0,0,17,3,A,0.12,AGUARDAR,DEFAULT,Nenhuma regra disparou; manter monitoramento. ...
2,SKU-002,53,84.0,32,6,999,A,0.18,COMPRAR,R3_COMPRAR_LEADTIME_ALTO,Estoque abaixo do ROP e lead time alto. Melhor...
3,SKU-003,25,55.0,27,6,5,C,0.18,COMPRAR,R2_COMPRAR_ABAIXO_ROP,Estoque abaixo do ROP e variabilidade elevada....
4,SKU-004,82,55.0,44,9,3,B,0.30,AGUARDAR,DEFAULT,Nenhuma regra disparou; manter monitoramento. ...
5,SKU-005,109,97.0,48,13,10,C,0.09,AGUARDAR,DEFAULT,Nenhuma regra disparou; manter monitoramento. ...


## 8) Desafio (atividade): implemente duas regras novas
1. `R0_INVALIDO` (prioridade 0): se `rop == 0` → ação `REVISAR_DADOS`
2. `R4_COMPRAR_MARGEM_ALTA` (prioridade 4): se `margem >= 0.25` e `estoque < rop` → `COMPRAR`
3. Compare `value_counts()` antes/depois.


### Comparação do `value_counts()` antes e depois da implementação de `R4_COMPRAR_MARGEM_ALTA`

**Antes da implementação de `R4_COMPRAR_MARGEM_ALTA` (apenas `R0_INVALIDO` estava ativa):**

| acao     | count |
|----------|-------|
| AGUARDAR | 43    |
| COMPRAR  | 11    |
| URGENTE  | 6     |

**Depois da implementação de `R4_COMPRAR_MARGEM_ALTA`:**

| acao     | count |
|----------|-------|
| AGUARDAR | 41    |
| COMPRAR  | 13    |
| URGENTE  | 6     |

**Análise:**

* O número de ações `COMPRAR` aumentou em 2 (de 11 para 13).  
* O número de ações `AGUARDAR` diminuiu em 2 (de 43 para 41).  
* A quantidade de ações `URGENTE` permaneceu a mesma.  

Essa mudança indica que a regra `R4_COMPRAR_MARGEM_ALTA` identificou com sucesso dois itens adicionais que atenderam aos seus critérios para a ação `COMPRAR`, alterando assim a distribuição geral das decisões.

### Regra R0_INVALIDO

-   **Nome:** `R0_INVALIDO`
-   **Prioridade:** 0
-   **Condição:** Se o ROP (Ponto de Ressuprimento) for igual a 0.
-   **Ação:** `REVISAR_DADOS`
-   **Justificativa:** ROP igual a zero indica dado inválido ou faltante. Necessário revisar.

### Regra R4_COMPRAR_MARGEM_ALTA

-   **Nome:** `R4_COMPRAR_MARGEM_ALTA`
-   **Prioridade:** 4
-   **Condição:** Se a margem de lucro for maior ou igual a 0.25 E o estoque for menor que o ROP.
-   **Ação:** `COMPRAR`
-   **Justificativa:** Margem de lucro alta e estoque abaixo do ROP. Comprar para garantir disponibilidade.